In [2]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


GPU available: True
Device name: Tesla T4


In [3]:
!pip install kagglehub
import kagglehub

path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fake-and-real-news-dataset' dataset.
Path to dataset files: /kaggle/input/fake-and-real-news-dataset


In [4]:
import pandas as pd
import os

print(os.listdir(path))

['True.csv', 'Fake.csv']


In [5]:
fake_df = pd.read_csv(os.path.join(path, "Fake.csv"))
true_df = pd.read_csv(os.path.join(path, "True.csv"))

fake_df["label"] = 0
true_df["label"] = 1

df = pd.concat([fake_df, true_df], axis=0, ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

import re
def strip_source_tag(text):
    return re.sub(r'^.*?\(Reuters\)\s*-\s*', '', text)

df["text"] = df["text"].apply(strip_source_tag)

print(df.shape)
df["label"].value_counts()

(44898, 5)


,count
label,
0,23481
1,21417


In [6]:
from sklearn.model_selection import train_test_split

X = df["text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

Train size: 35918
Test size: 8980


In [7]:
!pip install transformers datasets -q

In [8]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

# Quick test to see what tokenization looks like
sample = tokenizer("This is a test sentence about fake news.", truncation=True, padding=True)
print(sample)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

{'input_ids': [101, 2023, 2003, 1037, 3231, 6251, 2055, 8275, 2739, 1012, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [9]:
print(tokenizer.decode(sample["input_ids"]))

[CLS] this is a test sentence about fake news. [SEP]


In [10]:
train_encodings = tokenizer(
    list(X_train),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    list(X_test),
    truncation=True,
    padding=True,
    max_length=256
)

print("Train encodings ready. Example length:", len(train_encodings["input_ids"][0]))

Train encodings ready. Example length: 256


In [11]:
import torch

class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_encodings, list(y_train))
test_dataset = NewsDataset(test_encodings, list(y_test))

print("Train dataset size:", len(train_dataset))
print("Test dataset size:", len(test_dataset))

Train dataset size: 35918
Test dataset size: 8980


In [12]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

model = model.to("cuda")  # move model to GPU
print("Model loaded on GPU")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on GPU


In [13]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=50,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()


Step,Training Loss
50,0.303318
100,0.115181
150,0.093280
200,0.068895
250,0.049879
300,0.046393
350,0.045064
400,0.043778
450,0.017004
500,0.031349


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
predictions = trainer.predict(test_dataset)

import numpy as np
y_pred = np.argmax(predictions.predictions, axis=1)

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
model.save_pretrained("./truthlens_distilbert")
tokenizer.save_pretrained("./truthlens_distilbert")

print("Model and tokenizer saved")

In [ ]:
!zip -r truthlens_distilbert.zip truthlens_distilbert

In [ ]:
from google.colab import files
files.download("truthlens_distilbert.zip")

In [28]:
from huggingface_hub import login
login()


In [29]:
model.push_to_hub("truthlens-distilbert")
tokenizer.push_to_hub("truthlens-distilbert")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...v9umblq/model.safetensors:   2%|1         | 4.25MB /  268MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/VinnSmoke/truthlens-distilbert/commit/f05ba3ff946afd341aa5a65a1585bd256eea9a81', commit_message='Upload tokenizer', commit_description='', oid='f05ba3ff946afd341aa5a65a1585bd256eea9a81', pr_url=None, repo_url=RepoUrl('https://huggingface.co/VinnSmoke/truthlens-distilbert', endpoint='https://huggingface.co', repo_type='model', repo_id='VinnSmoke/truthlens-distilbert'), pr_revision=None, pr_num=None)